<a href="https://colab.research.google.com/github/SattamAltwaim/StarX/blob/main/experiments/14_assembly_sketch_rasterization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup: clone the StarX repo and the pinned TripoSR commit, install this
# notebook's dependencies, mount Drive, and report what machine we are on.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "14"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR = "/content/StarX"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
    TRIPOSR_DIR = "/content/TripoSR"
else:
    # off Colab the kernel's cwd is unpredictable (VS Code often starts in
    # $HOME) - walk up from the notebook location and the cwd to find the
    # repo, with ~/StarX as the final fallback
    def _find_repo():
        candidates = [globals().get("__vsc_ipynb_file__"), os.getcwd()]
        for start in candidates:
            if not start:
                continue
            path = os.path.abspath(
                os.path.dirname(start) if os.path.isfile(start) else start
            )
            while path != os.path.dirname(path):
                if os.path.exists(os.path.join(path, "starx", "pins.py")):
                    return path
                path = os.path.dirname(path)
        home_repo = os.path.join(os.path.expanduser("~"), "StarX")
        if os.path.exists(os.path.join(home_repo, "starx", "pins.py")):
            return home_repo
        raise RuntimeError(
            "could not locate the StarX repo - start Jupyter inside it "
            "or clone it to ~/StarX"
        )

    REPO_DIR = _find_repo()
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if not os.path.exists(TRIPOSR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git",
         TRIPOSR_DIR],
        check=True,
    )
subprocess.run(["git", "-C", TRIPOSR_DIR, "checkout", "-q", TRIPOSR_COMMIT], check=True)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
# .get, not [NOTEBOOK_ID]: the upstream starx/pins.py has no "14" entry
# (same situation as notebook 13 - this pin never made it into that shared
# file), so this must degrade to "nothing extra" instead of KeyError-ing.
if pins.PIP_PINS.get(NOTEBOOK_ID):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )
# trimesh (tsr imports it at module scope) and py7zr (the .7z archive) are
# needed regardless of what the upstream pins file says.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "trimesh", "py7zr"], check=True
)

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

# 14 - Assembly sketches into pixels

Same rasterizer as notebook 02 (`starx.fusion.load_design` +
`starx.rasterize.rasterize_design`), pointed at the Assembly (Joint) dataset
instead of the reconstruction one. Nothing about the rasterization logic
changes - it operates on a parsed `Design` (sketches + timeline), not on
which dataset the JSON came from. What's actually new here is entirely about
getting the right JSON in front of it:

1. get `j1.0.0.7z` onto fast local disk (same download logic notebook 13
   uses - that `.7z` handling isn't in the upstream repo, so it's redefined
   right here, same as there),
2. treat every per-body `.json` under `j1.0.0/joint/` as its own design -
   each body was modeled with its own sketch/extrude timeline, the same way
   a reconstruction design is. This is confirmed by the file layout notebook
   13 found: every `<id>_1` / `<id>_2` body ships its own full
   `json/obj/smt/step` set, not a shared one.
3. load one body, verify it actually parses as a `Design` with real
   sketches, then rasterize it and eyeball the result before trusting a
   batch run.

**This is the one real unknown notebook 13 didn't resolve:** whether a joint
dataset body's JSON uses the same `entities` / `timeline` / `Sketch` schema
the reconstruction dataset does. `starx.fusion.load_design` was written and
validated against the reconstruction schema; it has never been checked
against this one. The load-and-inspect cell below is that check - if it
comes back with zero sketches or throws, the schema differs and the parser
needs adapting, not this notebook's approach to it.

No GPU needed.

In [ ]:
# Configuration - every tunable for this notebook lives here. Same
# rasterization defaults as notebook 02, since nothing about them is
# dataset-specific.
import dataclasses
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py7zr
from tqdm.auto import tqdm

from starx import fusion, rasterize, viz
from starx.config import StarXConfig

SMOKE = True   # True: smaller batch test for a fast pipeline check
SAMPLE_N = 60 if SMOKE else 300   # bodies in the batch test
SEED = 1337

# New (the assembly dataset), so defined here rather than imported from
# starx.config - that module, as cloned above, doesn't know about it yet.
ASSEMBLY_DATA_URL = (
    "https://fusion-360-gallery-dataset.s3.us-west-2.amazonaws.com/"
    "assembly/j1.0.0/j1.0.0.7z"
)

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX")
    if DRIVE is not None
    else os.path.join(REPO_DIR, "data", "StarX"),
    sketch_size=512,
    supersample=2,
    stroke_width=5,
    stroke_value=0,
    bg_value=128,
    margin=0.05,
    normalization_mode="shared",
    max_sketch_channels=6,
    include_construction=False,
    truncate_extra_sketches=True,
)
archive_drive_path = Path(cfg.drive_root) / "raw" / "j1.0.0.7z"
print("channels:", cfg.max_sketch_channels, " image:", cfg.sketch_size)
print("archive on Drive lands at:", archive_drive_path)

In [ ]:
# Get the archive onto fast local disk (and archived on Drive). Same order
# as notebook 13: already local -> copy from Drive -> download once from S3
# to local disk and back it up to Drive. Redefined here because this logic
# isn't in the upstream starx.colab (hardcoded to r1.0.1.zip / zipfile there).
def ensure_assembly_archive_local(url, drive_path, local_dir):
    local_archive = Path(local_dir) / "j1.0.0.7z"
    if local_archive.exists():
        return local_archive
    if drive_path.exists():
        return scolab.copy_with_progress(drive_path, local_archive)
    scolab.download_with_progress(url, local_archive)
    if Path(cfg.drive_root).exists():
        scolab.copy_with_progress(local_archive, drive_path)
    return local_archive


archive_local_dir = "/content" if IN_COLAB else os.path.join(REPO_DIR, "data")
archive_path = ensure_assembly_archive_local(
    ASSEMBLY_DATA_URL, archive_drive_path, archive_local_dir
)
size_gib = archive_path.stat().st_size / 2**30
print(f"archive ready at {archive_path} ({size_gib:.2f} GiB)")

In [ ]:
# Every per-body json under j1.0.0/joint/ is its own design, the same unit
# notebook 02 rasterizes for the reconstruction dataset - confirmed by the
# file layout notebook 13 found: each body (the "_1" / "_2" suffix) ships
# its own full json/obj/smt/step set, not a shared one.
with py7zr.SevenZipFile(archive_path, mode="r") as _archive:
    names = _archive.getnames()

json_names = [
    n for n in names if n.startswith("j1.0.0/joint/") and n.endswith(".json")
]
design_ids = sorted(os.path.splitext(os.path.basename(n))[0] for n in json_names)
print(f"{len(design_ids)} bodies with a .json under j1.0.0/joint/")

In [ ]:
# Pull one body's json out and see whether it parses as a Design at all -
# the actual unknown this notebook exists to resolve. Everything this
# notebook writes lands on Drive (under assembly_sketch_check/), not the
# Colab session's local disk - slower per file than /content, but this
# notebook only ever touches a handful of files (one sample + one batch),
# never the full dataset, so that cost is negligible.
OUTPUT_DIR = Path(cfg.drive_root) / "assembly_sketch_check"
SAMPLE_DIR = OUTPUT_DIR / "sample"
BATCH_DIR = OUTPUT_DIR / "batch"
FIGURE_DIR = OUTPUT_DIR / "figures"
for _d in (SAMPLE_DIR, BATCH_DIR, FIGURE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)
SAMPLE_ID = rng.choice(design_ids)
sample_member = f"j1.0.0/joint/{SAMPLE_ID}.json"
with py7zr.SevenZipFile(archive_path, mode="r") as _archive:
    _archive.extract(path=SAMPLE_DIR, targets=[sample_member])

design = fusion.load_design(SAMPLE_DIR / sample_member, design_id=SAMPLE_ID)
for s in design.sketches:
    print(f"{s.name}: {len(s.curves)} curves, timeline index {s.timeline_index}")
print(f"extrudes: {design.n_extrudes}")
print(f"parser warnings: {design.warnings or 'none'}")
if not design.sketches:
    print(
        "\nNO SKETCHES PARSED - the schema likely differs from the "
        "reconstruction dataset's. Inspect design.raw (the loaded dict) "
        "directly before trusting anything below this cell."
    )

In [ ]:
# If that parsed cleanly, look at the first sketch as polylines - same
# sanity check notebook 02 does.
if design.sketches:
    sketch = design.sketches[0]
    palette = plt.get_cmap("tab10").colors
    type_colors, handles = {}, {}
    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    for curve in sketch.curves.values():
        poly = fusion.sample_curve(curve, sketch.points, cfg.include_construction)
        if poly is None:
            continue
        ctype = curve["type"]
        color = type_colors.setdefault(ctype, palette[len(type_colors)])
        (line,) = ax.plot(poly[:, 0], poly[:, 1], color=color, linewidth=2)
        handles[ctype] = line
    ax.set_aspect("equal")
    ax.legend(handles.values(), handles.keys())
    ax.set_title(f"{SAMPLE_ID} / {sketch.name} - curves as polylines (cm)")
    fig.savefig(FIGURE_DIR / f"{SAMPLE_ID}_polylines.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# The full channel stack for this body: one channel per timeline sketch,
# blank padding after the last one - exactly notebook 02's rasterize_design.
if design.sketches:
    stack, meta = rasterize.rasterize_design(design, cfg)
    fig = viz.show_sketch_stack(stack, meta, title=design.design_id)
    fig.savefig(FIGURE_DIR / f"{SAMPLE_ID}_stack.png", dpi=150, bbox_inches="tight")
    plt.show()
    print({k: meta[k] for k in ("n_sketches_total", "truncated", "blank_channels")})

In [ ]:
# Batch-test across a random sample of bodies straight from the archive.
# Extract all their jsons in one archive pass (into BATCH_DIR on Drive,
# set up two cells above), then rasterize each, catching failures per body
# instead of letting one bad design kill the run.
batch_ids = rng.sample(design_ids, min(SAMPLE_N, len(design_ids)))
batch_members = [f"j1.0.0/joint/{d}.json" for d in batch_ids]
with py7zr.SevenZipFile(archive_path, mode="r") as _archive:
    _archive.extract(path=BATCH_DIR, targets=batch_members)

metas, failures, gallery = [], [], []
for design_id, member in tqdm(list(zip(batch_ids, batch_members)), desc="rasterizing"):
    try:
        d = fusion.load_design(BATCH_DIR / member, design_id=design_id)
        stack_i, meta_i = rasterize.rasterize_design(d, cfg)
    except Exception as error:
        failures.append({"design_id": design_id, "error": repr(error)})
        continue
    metas.append(meta_i)
    if len(gallery) < 8 and not meta_i["all_blank"]:
        gallery.append((design_id, stack_i, meta_i))

print(f"clean rasterization: {len(metas) / len(batch_ids):.1%}")
pd.DataFrame(failures).head(10) if failures else print("no failures")

In [ ]:
# A gallery of channel strips from the batch - the model's-eye view of
# several different assembly bodies. Saved to Drive alongside the others.
fig, axes = plt.subplots(
    len(gallery), 1, figsize=(2.0 * cfg.max_sketch_channels, 2.15 * len(gallery))
)
for ax, (design_id, stack_g, meta_g) in zip(np.atleast_1d(axes), gallery):
    ax.imshow(rasterize.stack_to_strip(stack_g), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{design_id}  ({meta_g['n_sketches_total']} sketches)", fontsize=9)
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "batch_gallery.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Edge-case stats over the batch, same shape as notebook 02's.
meta_df = pd.DataFrame(metas)
if len(meta_df):
    print(f"designs truncated in batch: {meta_df['truncated'].mean():.1%}")
    print(
        f"designs with nothing drawable: "
        f"{int(meta_df['all_blank'].sum())} of {len(meta_df)}"
    )

## Takeaways

- If the load-and-inspect cell found real sketches and the batch clean-rate
  is high, `starx.fusion` / `starx.rasterize` need zero changes for the
  assembly dataset - the schema matches the reconstruction dataset closely
  enough that this is purely a "point it at different data" job, not a new
  parser.
- If clean rasterization is low, or the first cell found zero sketches,
  check the `failures` list above - it has the actual exception per body,
  which is the fastest way to tell a genuinely different schema apart from
  just a handful of unsupported curve types (the spline NURBS path,
  ellipses, etc. are the usual suspects, per notebook 02's own coverage
  notes).
- **Not resolved here:** whether to rasterize each body independently (as
  done above) or feed the model both bodies of a joint pair together
  somehow. That's a modeling decision for later, not a data question this
  notebook answers.

Next: once this is validated, the notebook 03 equivalent for this data
(ground-truth renders of each body's mesh + the full-dataset shard build)
can start - which still needs the per-body mesh compositing question from
notebook 13 settled first.